In [ ]:
import os
import shutil
import sys
from unittest.mock import MagicMock

from datasets import load_dataset
import torch
from transformers import TrainingArguments
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
dataset = load_dataset("json", data_files="../golden-dataset/dpo_training_data.jsonl", split="train")
unsafe_dataset = load_dataset("json", data_files="../golden-dataset/unsafe_dpo_training_data.jsonl", split="train")

In [ ]:
sys.modules['llm_blender'] = MagicMock()
sys.modules['mergekit'] = MagicMock()
sys.modules['mergekit.config'] = MagicMock()
sys.modules['mergekit.merge'] = MagicMock()

## FUNCTIONS

In [ ]:
def get_model_and_tokenizer(model_name, max_seq_length = 1024, dtype = None, load_in_4bit = True):
    base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = None,
    disable_log_stats = True,
    )
    return base_model, tokenizer

In [ ]:
def get_peft_model(base_model):
    peft_model = FastLanguageModel.get_peft_model(
        base_model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj",],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = False,
        loftq_config = None,
    )
    return peft_model

In [ ]:
def sft_formatting_data(tokenizer, dataset):
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = "chatml",
    )

    def formatting_prompts_func(examples):
        convos = []
        for system, prompt, chosen in zip(examples["system"], examples["prompt"], examples["chosen"]):
            convo = [
                {"role": "system", "content": system},
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": chosen}
            ]
            convos.append(convo)
        texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
        return { "text" : texts }

    sft_dataset = dataset.map(formatting_prompts_func, batched = True)
    return sft_dataset

In [ ]:
def dpo_formatting_data(tokenizer, dataset):
    tokenizer = get_chat_template(tokenizer, chat_template = "chatml")

    def format_dpo(examples):
        prompts = []
        chosens = []
        rejecteds = []
        for sys_prompt, p, c, r in zip(examples["system"], examples["prompt"], examples["chosen"], examples["rejected"]):
            convo = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": p}]
            prompt_str = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=True)

            chosen_str = c + "<|im_end|>"
            rejected_str = str(r) + "<|im_end|>"

            prompts.append(prompt_str)
            chosens.append(chosen_str)
            rejecteds.append(rejected_str)

        return {"prompt": prompts, "chosen": chosens, "rejected": rejecteds}

    dpo_dataset = dataset.map(format_dpo, batched=True)
    return dpo_dataset

In [ ]:
def get_sft_trainer(peft_model, tokenizer, sft_dataset):
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/temp/Qwen3-Persona-SFT",
        max_seq_length = 1024,           
        dataset_text_field = "text",
        padding_free = False,
    )

    trainer = SFTTrainer(
        model = peft_model,
        tokenizer = tokenizer,
        train_dataset = sft_dataset,
        args = args, 
        dataset_num_proc = 2,
        packing = False,
    )
    return trainer

In [ ]:
def get_dpo_trainer(model, tokenizer, dpo_dataset, max_seq_length = 1024):
    training_args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 5e-5,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/temp/Qwen3-Persona-DPO",
        report_to = "none",
    )

    trainer = DPOTrainer(
        model = model,
        ref_model = None,
        processing_class = tokenizer, 
        beta = 0.1,
        train_dataset = dpo_dataset,
        max_length = max_seq_length,
        max_prompt_length = 768,
        args = training_args,
    )
    return trainer

## Training SFT(safe+unsafe)

In [ ]:
print("1. Downloading Qwen 3 4B Instruct...")
base_model, tokenizer = get_model_and_tokenizer(MODEL_NAME)

In [ ]:
print("2. Attaching LoRA Adapters...")
sft_model = get_peft_model(base_model)

In [ ]:
print("3. Loading and Formatting Dataset...")
sft_dataset = sft_formatting_data(tokenizer, dataset)

In [ ]:
print("4. Configuring Ultra-Low VRAM Trainer and Starting Training...")
trainer = get_sft_trainer(sft_model, tokenizer, sft_dataset)
trainer_stats = trainer.train()

print("SFT COMPLETE!")

In [ ]:
print("5.Exporting model to GGUF format...")

sft_model.save_pretrained_gguf(
    "/Qwen3-4B-SFT",
    tokenizer,
    quantization_method = "q4_k_m"
)
print("Export complete!")

## Training SFT-DPO(safe+unsafe)

In [ ]:
print("1. Loading Warmed-Up SFT Model...")
sft_model_path = "/temp/Qwen3-Persona-SFT/checkpoint-1620"  # <-- LAST CHECKPOINT!
sft_dpo_model, tokenizer = get_model_and_tokenizer(sft_model_path)

In [ ]:
print("2. Formatting DPO Dataset...")
dpo_dataset = dpo_formatting_data(tokenizer, dataset)

In [ ]:
print("4. Configuring DPO Trainer and Starting Training...")
trainer = get_dpo_trainer(sft_dpo_model, tokenizer, dpo_dataset)
trainer.train()

print("DPO COMPLETE!")

In [ ]:
print("4. Exporting model to GGUF format...")
sft_dpo_model.save_pretrained_gguf(
    "/temp/Qwen-3B-FINAL",
    tokenizer,
    quantization_method = "q4_k_m",
)

shutil.copytree("/temp/Qwen-3B-FINAL_gguf", "/Qwen3-4B-SFT-DPO", dirs_exist_ok=True)

print("EXPORT COMPLETE!")


## Training DPO(safe+unsafe)

In [ ]:
print("1. Downloading Qwen 3 4B Instruct...")
base_model, tokenizer = get_model_and_tokenizer(MODEL_NAME)

In [ ]:
print("2. Attaching LoRA Adapters...")
dpo_model = get_peft_model(base_model)

In [ ]:
print("3. Formatting DPO Dataset...")
dpo_dataset = dpo_formatting_data(tokenizer, dataset)

In [ ]:
print("4. Configuring Modern DPO Trainer and Starting Training...")
trainer = get_dpo_trainer(dpo_model, tokenizer, dpo_dataset)
trainer.train()

print("DPO COMPLETE!")

In [ ]:
print("5. Exporting model to GGUF format...")
dpo_model.save_pretrained_gguf(
    "/temp/Qwen-3B-FINAL",
    tokenizer,
    quantization_method = "q4_k_m",
)

shutil.copytree("/temp/Qwen-3B-FINAL_gguf", "/Qwen3-4B-DPO", dirs_exist_ok=True)

print("EXPORT COMPLETE!")

## Training SFT-DPO(unsafe)

In [ ]:
print("1. Downloading Qwen 3 4B Instruct...")
base_model, tokenizer = get_model_and_tokenizer(MODEL_NAME)

In [ ]:
print("2. Attaching LoRA Adapters...")
sft_model = get_peft_model(base_model)

In [ ]:
print("3. Loading and Formatting Dataset...")
sft_unsafe_dataset = sft_formatting_data(tokenizer, unsafe_dataset)

In [ ]:
print("4. Configuring Ultra-Low VRAM Trainer and Starting Training...")
trainer = get_sft_trainer(sft_model, tokenizer, sft_unsafe_dataset)
trainer_stats = trainer.train()

print("SFT COMPLETE!")

In [ ]:
print("5. Loading Warmed-Up SFT Model...")
sft_model_path = "/temp/Qwen3-Persona-SFT/checkpoint-1620"  # <-- LAST CHECKPOINT!
sft_dpo_model, tokenizer = get_model_and_tokenizer(sft_model_path)

In [ ]:
print("6. Formatting DPO Dataset...")
dpo_unsafe_dataset = dpo_formatting_data(tokenizer, unsafe_dataset)

In [ ]:
print("7. Configuring DPO Trainer and Starting Training...")
trainer = get_dpo_trainer(sft_dpo_model, tokenizer, dpo_unsafe_dataset)
trainer.train()

print("DPO COMPLETE!")

In [ ]:
print("8. Exporting model to GGUF format...")
sft_dpo_model.save_pretrained_gguf(
    "/temp/Qwen-3B-FINAL",
    tokenizer,
    quantization_method = "q4_k_m",
)

shutil.copytree("/temp/Qwen-3B-FINAL_gguf", "/Qwen3-4B-SFT-DPO-unsafe", dirs_exist_ok=True)

print("EXPORT COMPLETE!")
